# 51 — A rubber ball in the blowdown pipe: the gas moves the ball, the ball shapes the flow

The rig of notebook 50: 1 L at 50 bar (absolute) and 20 °C, a 10 mm bore 100 mm long, a valve 20 mm down the pipe that
opens at t = 0. Now a **solid rubber ball (Ø 10 mm, 1100 kg/m³, 0.58 g)** sits tightly in the bore 5 mm after the valve.

- **The gas affects the ball**: 50 bar on its 78.5 mm² rear face is ~390 N on 0.58 g, tens of thousands of g.
- **The ball affects the flow**: it seals the bore, so it is a piston. Behind it the driver gas expands as it chases the
  ball; ahead of it a shock compresses the air and pushes it out of the exit. Nothing passes the ball until it leaves.

**CFD** (`sonic_ball.py`): OpenFOAM v2412 `rhoCentralFoam` on a moving mesh. The ball is a rigid 6-DoF body (axial
only) driven by the forces OpenFOAM integrates on its faces. The bore cells stretch behind it and squeeze ahead of it
(`displacementLayeredMotion`). The run ends as the ball reaches the exit.

**0-D gun model**: a lumped driver volume fed by a choked throat, a piston-driven shock ahead, ball friction.

**Bore sweep**: 5, 7, 10, 12, 15, 20 mm, the ball always the bore's size → ball exit speed vs pipe diameter.

In [ ]:
DEBUG = False                                  # real OpenFOAM (no mocks)
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)
RUN_CFD = True                                 # solves what is missing; finished cases are reused
import json, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import sonic_flow as sf, sonic_ball as sb
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.color": "#e4e3df", "axes.spines.top": False,
                     "axes.spines.right": False, "lines.linewidth": 2.0, "legend.frameon": False})
COL = {"air": "#2a78d6", "co2": "#eb6834"}; LAB = {"air": "Air", "co2": "CO$_2$"}
rig, ball, cond = sf.Rig(), sb.Ball(), sb.BALL_CONDITIONS
OUT = Path("output/sonic_ball"); RES = Path("sonic_ball_results"); RES.mkdir(exist_ok=True)

## 1. The design

In [ ]:
print(f"ball: Ø {ball.diameter} mm, {ball.density:.0f} kg/m³ (solid rubber), mass {ball.mass*1e3:.3f} g")
print(f"ball rear at x = {ball.rear_x} mm (valve seat at {rig.valve_x} mm), centre {ball.centre(rig)} mm; "
      f"travel to the exit {rig.pipe_length - ball.rear_x - ball.diameter:.0f} mm")
F0 = (cond.p0 - cond.p_amb) * rig.exit_area_m2
print(f"full container pressure on the ball: {F0:.0f} N -> {F0/ball.mass/9.81:,.0f} g")
cad = sb.make_cad(RES / "cad", rig, ball)
print(f"ball mass from the CAD volume: {cad['ball_mass_from_cad_g']:.4f} g")
from vegeta import dedalus
fig = dedalus.plot_views(dedalus.load_step(RES / "cad/ball_rig.step"))
fig.suptitle("Supply rig with the 10 mm rubber ball in the bore"); plt.show()

In [ ]:
mesh = sf.Mesh()                               # 0.5 mm in the bore, 10 cells across the radius
fig, ax = plt.subplots(figsize=(11, 3.4))
colors = {"container": "#b0aea6", "pipeBehindBall": "#2a78d6", "pipeAheadOfBall": "#eb6834", "ambient": "#d8d6cf"}
for x0, x1, r0, r1, nx, nr, ex, er, zone in sb.blocks(rig, mesh, ball):
    ax.add_patch(plt.Rectangle((x0, r0), x1 - x0, r1 - r0, color=colors[zone], alpha=0.55))
    if zone.startswith("pipe"):
        ax.text((x0 + x1) / 2, r1 + 0.6, f"{nx}×{nr}", ha="center", fontsize=7)
ax.add_patch(plt.Circle((ball.centre(rig), 0), ball.diameter / 2, color="k", alpha=0.8))
ax.axvline(rig.valve_x, color="k", ls="--", lw=1); ax.text(rig.valve_x + 0.5, 9, "valve", fontsize=8)
ax.set_xlim(-20, 130); ax.set_ylim(0, 12); ax.set_aspect("equal")
ax.set_xlabel("x [mm]"); ax.set_ylabel("r [mm]")
ax.set_title("Moving zones: blue stretches behind the ball, orange squeezes ahead; grey stands still", loc="left", fontsize=10)
plt.show()

## 2. 0-D gun model (instant)

In [ ]:
G = {g: {fr: sb.gun(sf.GASES[g], rig, ball=ball, friction=fr) for fr in (0.0, 10.0, 50.0)} for g in ("air", "co2")}
for g, runs in G.items():
    for fr, r in runs.items():
        print(f"{g:4s} friction {fr:4.0f} N: exit after {r['t'][-1]*1e3:.3f} ms at {r['u'][-1]:.1f} m/s, "
              f"peak {r['a'].max()/9.81:,.0f} g, air ahead up to {r['p_ahead'].max()/1e5:.2f} bar")

## 3. CFD: the coupled ball run, both gases (0.5 mm mesh)

In [ ]:
cases = {g: sb.solve(OUT / g, sf.GASES[g], rig, cond, mesh, ball, run=RUN_CFD) for g in ("air", "co2")}
H = {g: sb.histories(c) for g, c in cases.items()}
M = {g: sb.muzzle(sb.ball_track(c), ball, rig) for g, c in cases.items()}
for g, m in M.items():
    h = H[g]
    print(f"{g:4s}: ball leaves the pipe after {m['time']*1e3:.3f} ms at {m['speed']:.1f} m/s "
          f"(tracked to front = {m['front_at_stop_mm']:.1f} mm at {m['speed_at_stop']:.1f} m/s); "
          f"peak force {h['ball_force'].max():.0f} N, peak rear pressure {h['p_rear'].max()/1e5:.1f} bar")

### Ball: position, velocity, acceleration, force

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for g, h in H.items():
    t, x, u = h["ball_t"] * 1e3, h["ball_x"] * 1e3, h["ball_u"]
    F = np.interp(h["ball_t"], h["force_t"], h["ball_force"])
    axes[0][0].plot(t, x + ball.diameter / 2, color=COL[g], label=f"{LAB[g]} CFD")
    axes[0][1].plot(t, u, color=COL[g], label=f"{LAB[g]} CFD")
    axes[1][0].plot(t, F / ball.mass / 9.81 / 1e3, color=COL[g], label=f"{LAB[g]} CFD")
    axes[1][1].plot(t, F, color=COL[g], label=f"{LAB[g]} CFD")
    for fr, ls in ((0.0, "--"), (10.0, ":")):
        r = G[g][fr]
        lab = f"{LAB[g]} 0-D, friction {fr:.0f} N"
        axes[0][0].plot(r["t"] * 1e3, r["ball_x"] * 1e3 + ball.diameter / 2, color=COL[g], ls=ls, lw=1.4, label=lab)
        axes[0][1].plot(r["t"] * 1e3, r["u"], color=COL[g], ls=ls, lw=1.4, label=lab)
        axes[1][0].plot(r["t"] * 1e3, r["a"] / 9.81 / 1e3, color=COL[g], ls=ls, lw=1.4, label=lab)
        axes[1][1].plot(r["t"] * 1e3, r["force"], color=COL[g], ls=ls, lw=1.4, label=lab)
axes[0][0].axhline(rig.pipe_length, color="#52514e", ls="-.", lw=1); axes[0][0].text(0.01, rig.pipe_length + 1, "pipe exit", fontsize=8)
for ax, (title, unit) in zip(axes.flat, (("Ball front position", "mm"), ("Ball velocity", "m/s"),
                                          ("Ball acceleration", "1000 g"), ("Net gas force on the ball", "N"))):
    ax.set_title(title, loc="left"); ax.set_ylabel(unit)
for ax in axes[1]:
    ax.set_xlabel("time after the valve opens [ms]")
axes[0][1].legend(fontsize=7.5); fig.tight_layout(); plt.show()

### The flow around the ball: pressures, mass flows, exit gas velocity — with and without the ball

In [ ]:
plain = {}
for g in ("air", "co2"):                      # the same rig without the ball: production run if there, else the smoke test
    for p in (Path("output/sonic_flow") / g, Path("output/sonic_smoke") / g):
        if (p / "postProcessing/exitPlane").exists():
            plain[g] = (sf.histories(p), p.parent.name); break
t_end = max(h["ball_t"][-1] for h in H.values()) * 1e3
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for g, h in H.items():
    axes[0][0].plot(h["pball_t"] * 1e3, h["p_rear"] / 1e5, color=COL[g], label=f"{LAB[g]} behind the ball (max on its faces)")
    axes[0][0].plot(h["pball_t"] * 1e3, h["p_front"] / 1e5, color=COL[g], ls="--", label=f"{LAB[g]} ahead of the ball (min)")
    axes[0][1].plot(h["t"] * 1e3, h["inlet_mdot"], color=COL[g], label=f"{LAB[g]} leaving the container, with ball")
    axes[1][0].plot(h["t"] * 1e3, h["exit_U_area"], color=COL[g], label=f"{LAB[g]} with ball (air pushed out ahead of it)")
    axes[1][1].plot(h["t"] * 1e3, h["exit_mdot"], color=COL[g], label=f"{LAB[g]} with ball")
    if g in plain:
        q, src = plain[g]; k = q["t"] * 1e3 <= t_end
        axes[0][1].plot(q["t"][k] * 1e3, q["valve_mdot"][k], color=COL[g], ls=":", label=f"{LAB[g]} no ball ({src})")
        axes[1][0].plot(q["t"][k] * 1e3, q["exit_U_area"][k], color=COL[g], ls=":", label=f"{LAB[g]} no ball")
        axes[1][1].plot(q["t"][k] * 1e3, q["exit_mdot"][k], color=COL[g], ls=":", label=f"{LAB[g]} no ball")
for ax, (title, unit) in zip(axes.flat, (("Pressure on the ball", "bar"), ("Mass flow out of the container", "kg/s"),
                                          ("Exit gas velocity (area-averaged)", "m/s"), ("Mass flow at the pipe exit", "kg/s"))):
    ax.set_title(title, loc="left"); ax.set_ylabel(unit); ax.legend(fontsize=7.5)
for ax in axes[1]:
    ax.set_xlabel("time after the valve opens [ms]")
fig.tight_layout(); plt.show()

### Pressure along the axis vs time (a wave diagram), the ball's path drawn over it

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
for ax, (g, c) in zip(axes, cases.items()):
    times, rows = sb.axis_profiles(c, fields=("p",))
    xs = np.linspace(-0.02, 0.15, 400)
    P = np.full((len(times), len(xs)), np.nan)
    tr = sb.ball_track(c)
    for i, (t, r) in enumerate(zip(times, rows)):
        P[i] = np.interp(xs, r["x"], r["p"])
        xc = np.interp(t, tr["t"], tr["x"])
        P[i, np.abs(xs - xc) < ball.diameter * 0.5e-3] = np.nan          # the ball itself
    im = ax.pcolormesh(xs * 1e3, times * 1e3, P / 1e5, cmap="viridis", shading="nearest", vmin=0, vmax=50)
    ax.plot((tr["x"] - ball.diameter * 0.5e-3) * 1e3, tr["t"] * 1e3, color="w", lw=1.2)
    ax.plot((tr["x"] + ball.diameter * 0.5e-3) * 1e3, tr["t"] * 1e3, color="w", lw=1.2)
    ax.axvline(rig.valve_x, color="w", ls=":", lw=0.8); ax.axvline(rig.pipe_length, color="w", ls="--", lw=0.8)
    ax.set_title(f"{LAB[g]}: pressure on the axis (white: the ball)", loc="left"); ax.set_xlabel("x [mm]")
axes[0].set_ylabel("time [ms]")
fig.colorbar(im, ax=axes, label="p [bar]", shrink=0.85); plt.show()

### What the ball goes through

In [ ]:
rows = []
for g, h in H.items():
    F = h["ball_force"]; a = F.max() / ball.mass
    dp = (h["p_rear"] - h["p_front"]).max()
    rows.append((LAB[g], F.max(), a / 9.81, dp / 1e5, M[g]["speed"], 0.5 * ball.mass * M[g]["speed"] ** 2))
print(f"{'gas':6s} {'peak F [N]':>11s} {'peak a [g]':>11s} {'peak Δp [bar]':>14s} {'exit v [m/s]':>13s} {'KE [J]':>8s}")
for r in rows:
    print(f"{r[0]:6s} {r[1]:11.0f} {r[2]:11,.0f} {r[3]:14.1f} {r[4]:13.1f} {r[5]:8.1f}")
print("\nThe rubber sees up to ~50 bar (5 MPa) on its rear face against ~1-2 bar ahead. That is above the modulus of "
      "soft rubber (1-5 MPa): the real ball squashes axially and presses harder on the wall while it is driven. "
      "The CFD treats it as rigid.")

## 4. Bore sweep: ball exit speed vs pipe diameter

In [ ]:
SW = {}
for g in ("air", "co2"):
    try:
        SW[g] = sb.sweep(sf.GASES[g], run=RUN_CFD and g == "air")    # CO2 CFD only if it was solved beforehand
    except FileNotFoundError as e:
        print(f"{g}: CFD sweep not solved ({e})")
D = np.array(sb.SWEEP_DIAMETERS)
Z = {g: {fr: [sb.gun(sf.GASES[g], *sb.bore_setup(d)[0:1], ball=sb.bore_setup(d)[1], friction=fr) for d in D]
         for fr in (0.0, 10.0)} for g in ("air", "co2")}
for g, rows in SW.items():
    for r in rows:
        print(f"{g:4s} Ø{r['diameter_mm']:4.0f} mm  ball {r['ball_mass_g']:6.3f} g  exit speed {r['speed']:6.1f} m/s  after {r['time']*1e3:.3f} ms")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for g in ("air", "co2"):
    for fr, ls in ((0.0, "--"), (10.0, ":")):
        axes[0].plot(D, [r["u"][-1] for r in Z[g][fr]], color=COL[g], ls=ls, lw=1.5, marker="o", ms=4,
                     label=f"{LAB[g]} 0-D, friction {fr:.0f} N")
        axes[1].plot(D, [r["t"][-1] * 1e3 for r in Z[g][fr]], color=COL[g], ls=ls, lw=1.5, marker="o", ms=4)
    if g in SW:
        axes[0].plot(D, [r["speed"] for r in SW[g]], color=COL[g], marker="s", ms=8, label=f"{LAB[g]} CFD (coupled)")
        axes[1].plot(D, [r["time"] * 1e3 for r in SW[g]], color=COL[g], marker="s", ms=8, label=f"{LAB[g]} CFD")
        for d, r in zip(D, SW[g]):
            axes[0].annotate(f"{r['speed']:.0f}", (d, r["speed"]), textcoords="offset points", xytext=(6, 6), fontsize=8)
axes[0].set_title("Ball exit speed vs pipe diameter (ball = bore)", loc="left"); axes[0].set_ylabel("m/s")
axes[1].set_title("Time from valve opening to ball exit", loc="left"); axes[1].set_ylabel("ms")
for ax in axes:
    ax.set_xlabel("pipe (and ball) diameter [mm]"); ax.set_xticks(D)
axes[0].set_ylim(bottom=0); axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(RES / "ball_speed_vs_diameter.png", dpi=150); plt.show()

## 5. Reading it

*(filled in from the results below)*